In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 1.8: Recurrent Autoencoder (Sequential Data)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build an LSTM-based autoencoder for time-series data
#   - Understand WHY recurrent architecture preserves temporal order
#   - Visualise original vs reconstructed time-series overlays
#   - Apply to SGX financial anomaly / regime change detection
#   - Quantify portfolio drawdown reduction in S$ for a S$100M fund
#
# PREREQUISITES: 07_stacked_ae.py
# ESTIMATED TIME: ~20 min
#
# TASKS:
#   1. Generate synthetic sensor vibration data
#   2. Build LSTM encoder-decoder architecture
#   3. Train and visualise time-series reconstruction
#   4. Apply: SGX regime change detection with portfolio impact
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset



## THEORY — Temporal Order Matters


In [ ]:
# For sequential data, temporal order matters. An LSTM encoder reads
# the sequence step by step, building a summary (hidden state). The
# LSTM decoder unrolls the summary back into the original sequence.
#
# Analogy: Reading a book chapter vs reading the same words shuffled
# randomly. The LSTM encoder reads the chapter in order and produces
# a summary that captures the narrative arc. A flat MLP reads the
# shuffled words and has no sense of what comes before or after.
#
# WHY THIS MATTERS: Sensor data, financial returns, medical signals
# all have temporal structure. A vibration spike AFTER a speed change
# means something different than a spike BEFORE one.



## TASK 1 — Generate Sensor Data and Set Up Engines


Generate synthetic industrial vibration sensor data.


In [ ]:
conn, tracker, exp_name, registry, has_registry = setup_engines()

SEQ_LEN = 100
N_SERIES_TRAIN = 5000
N_SERIES_TEST = 500


def generate_sensor_data(n_samples, seq_len, seed=42):
    # TODO: Generate n_samples time series, each of length seq_len
    # Each series: sum of 2 sinusoids at random frequencies + noise
    # t = np.linspace(0, 4*pi, seq_len)
    # signal = amp1*sin(freq1*t+phase) + amp2*sin(freq2*t) + 0.1*noise
    # Normalise each series to [0, 1]
    ____


# TODO: Generate train and test sensor data
sensor_train = ____
sensor_test = ____

sensor_train_t = torch.tensor(sensor_train).to(device)
sensor_test_t = torch.tensor(sensor_test).to(device)
sensor_loader = DataLoader(TensorDataset(sensor_train_t), batch_size=128, shuffle=True)

print(
    f"Sensor data: {sensor_train.shape[0]} train, {sensor_test.shape[0]} test, seq_len={SEQ_LEN}"
)



## TASK 2 — Build and Train Recurrent AE


LSTM-based autoencoder for sequential data.


In [ ]:
class RecurrentAE(nn.Module):

    def __init__(self, seq_len: int, hidden_dim: int = 64, latent_dim: int = 16):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_dim = hidden_dim
        # TODO: Define layers:
        #   encoder_lstm: nn.LSTM(input_size=1, hidden_size=hidden_dim, batch_first=True)
        #   enc_to_latent: nn.Linear(hidden_dim, latent_dim)
        #   latent_to_dec: nn.Linear(latent_dim, hidden_dim)
        #   decoder_lstm: nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)
        #   output_layer: nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())
        self.encoder_lstm = ____
        self.enc_to_latent = ____
        self.latent_to_dec = ____
        self.decoder_lstm = ____
        self.output_layer = ____

    def forward(self, x):
        # TODO: Implement LSTM AE forward pass:
        # 1. x_seq = x.unsqueeze(-1)  — add feature dim
        # 2. Encode: _, (h_n, _) = self.encoder_lstm(x_seq)
        # 3. Compress: z = self.enc_to_latent(h_n.squeeze(0))
        # 4. Expand: dec_input = self.latent_to_dec(z).unsqueeze(1).repeat(1, seq_len, 1)
        # 5. Decode: dec_output, _ = self.decoder_lstm(dec_input)
        # 6. Output: x_hat = self.output_layer(dec_output).squeeze(-1)
        # Return (x_hat, z)
        ____


def recurrent_ae_loss(model, xb):
    # TODO: Forward, MSE loss. Return (loss, {})
    ____


print("\n" + "=" * 70)
print("  Recurrent AE — Time-Series (Sensor Data)")
print("=" * 70)
print("  LSTM encoder reads sequence -> latent -> LSTM decoder reconstructs.")

# TODO: Create RecurrentAE(SEQ_LEN, hidden_dim=64, latent_dim=LATENT_DIM) and train
recurrent_model = ____
recurrent_losses = ____



## TASK 3 — Visualise Time-Series Reconstruction


In [ ]:
# TODO: show_timeseries_reconstruction
____



### Checkpoint


In [ ]:
assert len(recurrent_losses) == EPOCHS
assert recurrent_losses[-1] < recurrent_losses[0]
# INTERPRETATION: The time-series overlay shows reconstructed (red
# dashed) tracking original (blue solid). Where lines diverge =
# patterns harder to compress. High reconstruction error = anomaly.
print("\n--- Checkpoint passed --- recurrent AE trained\n")

if has_registry:
    register_model(registry, "recurrent_ae", recurrent_model, recurrent_losses[-1])



## APPLY — SGX Financial Regime Change Detection


In [ ]:
# BUSINESS SCENARIO: You are a quantitative analyst at a Singapore
# hedge fund monitoring SGX equities for regime changes. Markets shift
# between calm and crisis states. Your PM asks: "Can we detect regime
# changes early enough to reduce portfolio drawdown?"

print("\n" + "=" * 70)
print("  APPLICATION: SGX Regime Change Detection (S$100M Fund)")
print("=" * 70)

# --- Generate SGX equity data ---
N_DAYS = 1500
N_STOCKS = 5
STOCK_NAMES = ["DBS", "OCBC", "Singtel", "CapitaLand", "Keppel"]
fin_rng = np.random.default_rng(42)

# TODO: Generate correlated daily returns for 5 SGX stocks
# base_returns, base_vols, correlation matrix, Cholesky decomposition
# Include 4 crisis periods with regime_labels
# Crisis: higher vol, negative drift
base_returns = np.array([0.08, 0.07, 0.04, 0.06, 0.05])
base_vols = np.array([0.18, 0.16, 0.22, 0.20, 0.24])
corr = np.array(
    [
        [1.00, 0.85, 0.45, 0.55, 0.50],
        [0.85, 1.00, 0.40, 0.50, 0.45],
        [0.45, 0.40, 1.00, 0.35, 0.30],
        [0.55, 0.50, 0.35, 1.00, 0.60],
        [0.50, 0.45, 0.30, 0.60, 1.00],
    ]
)
L = np.linalg.cholesky(corr)

daily_returns = np.zeros((N_DAYS, N_STOCKS), dtype=np.float32)
regime_labels = np.zeros(N_DAYS, dtype=np.int32)
crisis_periods = [
    (300, 360, "COVID Crash (Mar 2020)"),
    (600, 640, "Rate Hike Shock (2022)"),
    (900, 930, "Banking Stress (2023)"),
    (1200, 1230, "Geopolitical Crisis"),
]

# TODO: Fill daily_returns using correlated random draws
# Normal days: positive drift + normal vol
# Crisis days: negative drift + 3x vol
for day in range(N_DAYS):
    in_crisis = any(start <= day < end for start, end, _ in crisis_periods)
    z = fin_rng.standard_normal(N_STOCKS)
    corr_z = L @ z
    if in_crisis:
        regime_labels[day] = 1
        daily_returns[day] = ____
    else:
        daily_returns[day] = ____

prices = 100 * np.exp(np.cumsum(daily_returns, axis=0))
print(f"Generated {N_DAYS} trading days, {regime_labels.sum()} crisis days")

# --- Create sequences ---
FIN_SEQ_LEN = 20
# TODO: Create rolling features (returns + 5-day rolling vol)
# Normalise, create sequences of length FIN_SEQ_LEN
# Split into normal-only training set
rolling_vol = np.zeros_like(daily_returns)
for i in range(5, N_DAYS):
    rolling_vol[i] = daily_returns[i - 5 : i].std(axis=0)
features = np.concatenate([daily_returns, rolling_vol], axis=1)
N_FEATURES = features.shape[1]
feat_mean = features.mean(axis=0)
feat_std = features.std(axis=0)
feat_std[feat_std == 0] = 1.0
features_norm = ((features - feat_mean) / feat_std).astype(np.float32)

sequences, seq_labels, seq_days = [], [], []
for i in range(N_DAYS - FIN_SEQ_LEN):
    sequences.append(features_norm[i : i + FIN_SEQ_LEN])
    seq_labels.append(int(regime_labels[i : i + FIN_SEQ_LEN].any()))
    seq_days.append(i + FIN_SEQ_LEN)
sequences = np.array(sequences, dtype=np.float32)
seq_labels = np.array(seq_labels)
seq_days = np.array(seq_days)

normal_mask = seq_labels == 0
train_seqs = sequences[normal_mask][: int(normal_mask.sum() * 0.8)]
fin_train_tensor = torch.tensor(train_seqs, device=device)
fin_test_tensor = torch.tensor(sequences, device=device)
fin_train_loader = DataLoader(
    TensorDataset(fin_train_tensor), batch_size=128, shuffle=True
)

print(
    f"Training on {len(train_seqs)} normal-only sequences (shape: {FIN_SEQ_LEN}x{N_FEATURES})"
)


# --- LSTM Autoencoder ---
class LSTMEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True)

    def forward(self, x):
        _, (h, c) = self.lstm(x)
        return h, c


class LSTMDecoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, seq_len, n_layers=1):
        super().__init__()
        self.seq_len = seq_len
        self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h, c):
        x_repeated = x[:, -1:, :].repeat(1, self.seq_len, 1)
        out, _ = self.lstm(x_repeated, (h, c))
        return self.fc(out)


class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, seq_len):
        super().__init__()
        # TODO: Create encoder and decoder
        self.encoder = ____
        self.decoder = ____

    def forward(self, x):
        # TODO: Encode then decode
        ____


HIDDEN_DIM = 32
# TODO: Create LSTMAutoencoder, optimizer, criterion. Train 60 epochs.
fin_model = ____
fin_opt = ____
fin_criterion = nn.MSELoss()

print("\nTraining LSTM autoencoder on normal market periods...")
for epoch in range(60):
    fin_model.train()
    epoch_loss, n_batches = 0.0, 0
    for (batch,) in fin_train_loader:
        # TODO: Forward, loss, backprop
        ____
    if (epoch + 1) % 15 == 0:
        print(f"  Epoch {epoch+1:3d}/60: loss = {epoch_loss/n_batches:.6f}")

# --- Compute reconstruction errors ---
fin_model.eval()
with torch.no_grad():
    chunk_size = 512
    all_fin_errors = []
    for i in range(0, len(fin_test_tensor), chunk_size):
        chunk = fin_test_tensor[i : i + chunk_size]
        recon = fin_model(chunk)
        chunk_errors = ((chunk - recon) ** 2).mean(dim=(1, 2)).cpu().numpy()
        all_fin_errors.append(chunk_errors)
    recon_errors = np.concatenate(all_fin_errors)

normal_errors = recon_errors[seq_labels == 0]
crisis_errors = recon_errors[seq_labels == 1]
threshold = np.percentile(normal_errors, 95)
is_anomaly = recon_errors > threshold

print(f"\nNormal error: {normal_errors.mean():.4f}, Crisis: {crisis_errors.mean():.4f}")
print(f"Separation: {crisis_errors.mean() / normal_errors.mean():.1f}x")

# --- Visualisation 1: Price with anomaly overlay ---
# TODO: 2-row figure: DBS price with crisis shading, anomaly score with threshold
# Save to OUTPUT_DIR / "ex1_timeseries_anomaly_overlay.png"
fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={"height_ratios": [2, 1]})
____
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_timeseries_anomaly_overlay.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Visualisation 2: Event detection ---
# TODO: For each crisis period, check if anomaly was detected.
# Compute lead time (days of early warning).
# Bar chart of lead times per event.
# Save to OUTPUT_DIR / "ex1_timeseries_event_detection.png"
event_detected, event_lead_days = [], []
for start, end, name in crisis_periods:
    window_mask = (seq_days >= start - 30) & (seq_days <= end)
    window_anomalies = is_anomaly[window_mask]
    detected = window_anomalies.any()
    event_detected.append(detected)
    if detected:
        first_idx = np.where(window_anomalies)[0][0]
        first_day = seq_days[window_mask][first_idx]
        event_lead_days.append(max(start - first_day, 0))
    else:
        event_lead_days.append(0)

fig, ax = plt.subplots(figsize=(14, 6))
____
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_timeseries_event_detection.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Portfolio drawdown analysis ---
# TODO: Compare passive vs anomaly-adjusted portfolio
# When anomaly detected, reduce exposure to 20% for 5 days
# Compute cumulative returns, max drawdown for both strategies
# Save to OUTPUT_DIR / "ex1_timeseries_portfolio_drawdown.png"
PORTFOLIO_VALUE = 100_000_000
portfolio_returns = daily_returns.mean(axis=1)
passive_cum = np.cumprod(1 + portfolio_returns) * PORTFOLIO_VALUE

anomaly_by_day = np.zeros(N_DAYS, dtype=bool)
for i, day in enumerate(seq_days):
    if is_anomaly[i] and day < N_DAYS:
        anomaly_by_day[day : min(day + 5, N_DAYS)] = True
adjusted_returns = np.where(anomaly_by_day, portfolio_returns * 0.2, portfolio_returns)
adjusted_cum = np.cumprod(1 + adjusted_returns) * PORTFOLIO_VALUE

passive_peak = np.maximum.accumulate(passive_cum)
passive_dd = (passive_cum - passive_peak) / passive_peak
adjusted_peak = np.maximum.accumulate(adjusted_cum)
adjusted_dd = (adjusted_cum - adjusted_peak) / adjusted_peak

fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={"height_ratios": [2, 1]})
____
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_timeseries_portfolio_drawdown.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Business Impact ---
passive_worst_loss = PORTFOLIO_VALUE * abs(passive_dd.min())
adjusted_worst_loss = PORTFOLIO_VALUE * abs(adjusted_dd.min())
dollar_saved = passive_worst_loss - adjusted_worst_loss

print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — SGX Regime Detection (S$100M Fund)")
print("=" * 64)
print(f"\nEvents detected: {sum(event_detected)}/{len(crisis_periods)}")
for i, (_, _, name) in enumerate(crisis_periods):
    status = f"Detected {event_lead_days[i]}d early" if event_detected[i] else "MISSED"
    print(f"  {name}: {status}")
print(f"\nPassive max drawdown:   {passive_dd.min()*100:.1f}%")
print(f"Adjusted max drawdown:  {adjusted_dd.min()*100:.1f}%")
print(f"Capital preserved:      S${dollar_saved:,.0f}")
print(f"\nFinal portfolio value:")
print(f"  Passive:              S${passive_cum[-1]:,.0f}")
print(f"  Anomaly-adjusted:     S${adjusted_cum[-1]:,.0f}")
print("=" * 64)



## REFLECTION


[x] Built an LSTM encoder-decoder for time-series data
  [x] Understood temporal order preservation via recurrent architecture
  [x] Visualised original vs reconstructed vibration patterns
  [x] Applied to SGX regime change detection with early warning
  [x] Built portfolio anomaly-adjusted strategy
  [x] Quantified S$ capital preserved at worst drawdown

  KEY INSIGHT: The LSTM encoder reads a sequence step-by-step,
  building a compressed summary that captures temporal patterns.
  When the market enters a regime the model has never seen (crisis),
  reconstruction error spikes — the model is saying "this sequence
  does not match any pattern I learned from normal markets." That
  signal, days before the worst of the crash, is worth millions
  in avoided drawdown.

  Next: 09_vae.py introduces probabilistic latent spaces...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments (LSTM temporal flow)


In [ ]:
# LSTMs suffer a unique flavour of vanishing gradients: gradients
# don't vanish through DEPTH (we only have 1 recurrent layer here)
# but through TIME. The Blood Test here is especially informative.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    xb = batch[0] if isinstance(batch, (tuple, list)) else batch
    loss, _ = recurrent_ae_loss(m, xb)
    return loss


print("\n── Diagnostic Report (Recurrent AE) ──")
diag, findings = run_diagnostic_checkpoint(
    recurrent_model,
    sensor_loader,
    _diag_loss,
    title="Recurrent AE (LSTM)",
    n_batches=8,
    train_losses=recurrent_losses,
    show=False,
)

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
#   [!] Gradient flow (WARNING): Hidden-to-hidden gradient
#       attenuation at 'encoder.weight_hh_l0' — RMS = 3.6e-05.
#       Gradient flowing BACKWARD through time across 20 steps
#       shrinks by ~0.8^20 ≈ 1%. LSTM gating helps but does
#       not eliminate the pattern.
#   [✓] Saturation   (HEALTHY): max |tanh| = 0.87 on
#       encoder cell state (below 0.99 saturation flag).
#       Input/forget/output gates activations in [0.2, 0.8].
#   [✓] Loss trend    (HEALTHY): train slope -6.2e-04/epoch.
#       Slower than dense AEs — sequence reconstruction is
#       intrinsically harder than static image.



## Final train loss: ~0.014 after 10 epochs, sequence_length=20.

STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:

 [BLOOD TEST — VANISHING THROUGH TIME] RMS 3.6e-05 on
    weight_hh_l0 reveals the RNN-specific vanishing pattern:
    gradients flow BACKWARD through T=20 timesteps, and each
    backprop step multiplies by roughly the hidden-state
    Jacobian. LSTM's gating keeps this above the <1e-6 floor
    that would kill a vanilla RNN (demonstrated in ex_3/01),
    but the attenuation is intrinsic to BPTT. Slide 5L
    covers the Hochreiter 1991 analysis.
    >> Prescription: (a) Shorten sequence_length to 10 if
       gradient drops below 1e-5. (b) Switch to GRU (simpler
       gating, often similar performance — see ex_3/03).
       (c) Add attention or transformer blocks for very long
       sequences (ex_4).

 [X-RAY — LSTM-SPECIFIC SATURATION] Dead-neuron % is NOT the
    right X-Ray for LSTMs. Tanh and sigmoid saturate
    (output → ±1 or 0/1) rather than die. The instrument
    reports |tanh| statistics instead. 0.87 max is healthy;
    >0.99 would signal gate saturation — gates stuck open
    or closed, no information gating, degenerating into a
    linear RNN.
    >> Prescription: Gradient clipping at max_norm=5.0 +
       weight decay 1e-5 on recurrent matrices keeps
       activations away from saturation.

 [STETHOSCOPE] Slope -6.2e-04/epoch is slower than 02
    undercomplete (~-1.5e-3/epoch) on static images. This
    is the price of temporal modelling — each step depends
    on the prior step's hidden state, so the error signal
    is temporally correlated and harder to exploit.
    >> Prescription: No fix. If you need faster convergence,
       consider teacher forcing (feed ground-truth prior
       step during training) — standard RNN trick.

 FIVE-INSTRUMENT TAKEAWAY: recurrent AE introduces the
 TIME DIMENSION to the diagnostic vocabulary. The Blood
 Test now reads "across timesteps", the X-Ray shifts from
 "dead" to "saturated". These same readings recur in ex_3
 RNN variants (scaled up) and in ex_4 transformers (where
 attention replaces recurrence — different mechanism,
 same long-range gradient challenge).


## TASK 3 — Visualise Time-Series Reconstruction


In [ ]:
show_timeseries_reconstruction(
    recurrent_model,
    sensor_test_t,
    "Recurrent AE — Sensor Vibration Reconstruction",
)



### Checkpoint


In [ ]:
assert len(recurrent_losses) == EPOCHS
assert recurrent_losses[-1] < recurrent_losses[0]
# INTERPRETATION: The time-series overlay shows reconstructed (red
# dashed) tracking original (blue solid). Where lines diverge =
# patterns harder to compress. High reconstruction error = anomaly.
print("\n--- Checkpoint passed --- recurrent AE trained\n")

if has_registry:
    register_model(registry, "recurrent_ae", recurrent_model, recurrent_losses[-1])



## APPLY — SGX Financial Regime Change Detection


In [ ]:
# BUSINESS SCENARIO: You are a quantitative analyst at a Singapore
# hedge fund monitoring SGX equities for regime changes. Markets shift
# between calm and crisis states. Your PM asks: "Can we detect regime
# changes early enough to reduce portfolio drawdown?"

print("\n" + "=" * 70)
print("  APPLICATION: SGX Regime Change Detection (S$100M Fund)")
print("=" * 70)

# --- Generate SGX equity data ---
N_DAYS = 1500
N_STOCKS = 5
STOCK_NAMES = ["DBS", "OCBC", "Singtel", "CapitaLand", "Keppel"]
fin_rng = np.random.default_rng(42)

base_returns = np.array([0.08, 0.07, 0.04, 0.06, 0.05])
base_vols = np.array([0.18, 0.16, 0.22, 0.20, 0.24])
corr = np.array(
    [
        [1.00, 0.85, 0.45, 0.55, 0.50],
        [0.85, 1.00, 0.40, 0.50, 0.45],
        [0.45, 0.40, 1.00, 0.35, 0.30],
        [0.55, 0.50, 0.35, 1.00, 0.60],
        [0.50, 0.45, 0.30, 0.60, 1.00],
    ]
)
L = np.linalg.cholesky(corr)

daily_returns = np.zeros((N_DAYS, N_STOCKS), dtype=np.float32)
regime_labels = np.zeros(N_DAYS, dtype=np.int32)
crisis_periods = [
    (300, 360, "COVID Crash (Mar 2020)"),
    (600, 640, "Rate Hike Shock (2022)"),
    (900, 930, "Banking Stress (2023)"),
    (1200, 1230, "Geopolitical Crisis"),
]

for day in range(N_DAYS):
    in_crisis = any(start <= day < end for start, end, _ in crisis_periods)
    z = fin_rng.standard_normal(N_STOCKS)
    corr_z = L @ z
    if in_crisis:
        regime_labels[day] = 1
        daily_returns[day] = (
            -base_returns * 2.0 / 252 + base_vols * 3.0 / np.sqrt(252) * corr_z
        )
    else:
        daily_returns[day] = base_returns / 252 + base_vols / np.sqrt(252) * corr_z

prices = 100 * np.exp(np.cumsum(daily_returns, axis=0))
print(f"Generated {N_DAYS} trading days, {regime_labels.sum()} crisis days")

# --- Create sequences ---
FIN_SEQ_LEN = 20
rolling_vol = np.zeros_like(daily_returns)
for i in range(5, N_DAYS):
    rolling_vol[i] = daily_returns[i - 5 : i].std(axis=0)
features = np.concatenate([daily_returns, rolling_vol], axis=1)
N_FEATURES = features.shape[1]
feat_mean = features.mean(axis=0)
feat_std = features.std(axis=0)
feat_std[feat_std == 0] = 1.0
features_norm = ((features - feat_mean) / feat_std).astype(np.float32)

sequences, seq_labels, seq_days = [], [], []
for i in range(N_DAYS - FIN_SEQ_LEN):
    sequences.append(features_norm[i : i + FIN_SEQ_LEN])
    seq_labels.append(int(regime_labels[i : i + FIN_SEQ_LEN].any()))
    seq_days.append(i + FIN_SEQ_LEN)
sequences = np.array(sequences, dtype=np.float32)
seq_labels = np.array(seq_labels)
seq_days = np.array(seq_days)

normal_mask = seq_labels == 0
train_seqs = sequences[normal_mask][: int(normal_mask.sum() * 0.8)]
fin_train_tensor = torch.tensor(train_seqs, device=device)
fin_test_tensor = torch.tensor(sequences, device=device)
fin_train_loader = DataLoader(
    TensorDataset(fin_train_tensor), batch_size=128, shuffle=True
)

print(
    f"Training on {len(train_seqs)} normal-only sequences (shape: {FIN_SEQ_LEN}x{N_FEATURES})"
)


# --- LSTM Autoencoder ---
class LSTMEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True)

    def forward(self, x):
        _, (h, c) = self.lstm(x)
        return h, c


class LSTMDecoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, seq_len, n_layers=1):
        super().__init__()
        self.seq_len = seq_len
        self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h, c):
        x_repeated = x[:, -1:, :].repeat(1, self.seq_len, 1)
        out, _ = self.lstm(x_repeated, (h, c))
        return self.fc(out)


class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, seq_len):
        super().__init__()
        self.encoder = LSTMEncoder(input_dim, hidden_dim)
        self.decoder = LSTMDecoder(input_dim, hidden_dim, input_dim, seq_len)

    def forward(self, x):
        h, c = self.encoder(x)
        return self.decoder(x, h, c)


HIDDEN_DIM = 32
fin_model = LSTMAutoencoder(N_FEATURES, HIDDEN_DIM, FIN_SEQ_LEN).to(device)
fin_opt = torch.optim.Adam(fin_model.parameters(), lr=1e-3)
fin_criterion = nn.MSELoss()

print("\nTraining LSTM autoencoder on normal market periods...")
for epoch in range(60):
    fin_model.train()
    epoch_loss, n_batches = 0.0, 0
    for (batch,) in fin_train_loader:
        recon = fin_model(batch)
        loss = fin_criterion(recon, batch)
        fin_opt.zero_grad()
        loss.backward()
        fin_opt.step()
        epoch_loss += loss.item()
        n_batches += 1
    if (epoch + 1) % 15 == 0:
        print(f"  Epoch {epoch+1:3d}/60: loss = {epoch_loss/n_batches:.6f}")

# --- Compute reconstruction errors ---
fin_model.eval()
with torch.no_grad():
    chunk_size = 512
    all_fin_errors = []
    for i in range(0, len(fin_test_tensor), chunk_size):
        chunk = fin_test_tensor[i : i + chunk_size]
        recon = fin_model(chunk)
        chunk_errors = ((chunk - recon) ** 2).mean(dim=(1, 2)).cpu().numpy()
        all_fin_errors.append(chunk_errors)
    recon_errors = np.concatenate(all_fin_errors)

normal_errors = recon_errors[seq_labels == 0]
crisis_errors = recon_errors[seq_labels == 1]
threshold = np.percentile(normal_errors, 95)
is_anomaly = recon_errors > threshold

print(f"\nNormal error: {normal_errors.mean():.4f}, Crisis: {crisis_errors.mean():.4f}")
print(f"Separation: {crisis_errors.mean() / normal_errors.mean():.1f}x")

# --- Visualisation 1: Price with anomaly overlay ---
fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={"height_ratios": [2, 1]})
days = np.arange(N_DAYS)
axes[0].plot(days, prices[:, 0], color="#1565C0", linewidth=1.2, label="DBS Price")
for start, end, name in crisis_periods:
    axes[0].axvspan(start, end, alpha=0.15, color="#F44336", label=name)
axes[0].set_ylabel("Price (S$)")
axes[0].set_title("DBS Group — Price with Market Regime Detection", fontsize=14)
axes[0].legend(fontsize=9, loc="upper left", ncol=2)
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(
    seq_days, 0, recon_errors, alpha=0.4, color="#9E9E9E", label="Anomaly Score"
)
axes[1].fill_between(
    seq_days,
    0,
    recon_errors,
    where=is_anomaly,
    alpha=0.7,
    color="#F44336",
    label="Detected Anomaly",
)
axes[1].axhline(
    threshold,
    color="#FF9800",
    linestyle="--",
    linewidth=1.5,
    label=f"Threshold = {threshold:.4f}",
)
axes[1].set_ylabel("Reconstruction Error")
axes[1].set_xlabel("Trading Day")
axes[1].set_title("LSTM-AE Anomaly Score", fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_timeseries_anomaly_overlay.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Visualisation 2: Event detection ---
event_detected, event_lead_days = [], []
for start, end, name in crisis_periods:
    window_mask = (seq_days >= start - 30) & (seq_days <= end)
    window_anomalies = is_anomaly[window_mask]
    detected = window_anomalies.any()
    event_detected.append(detected)
    if detected:
        first_idx = np.where(window_anomalies)[0][0]
        first_day = seq_days[window_mask][first_idx]
        event_lead_days.append(max(start - first_day, 0))
    else:
        event_lead_days.append(0)

fig, ax = plt.subplots(figsize=(14, 6))
event_names = [name for _, _, name in crisis_periods]
colors = ["#4CAF50" if d else "#F44336" for d in event_detected]
ax.barh(range(len(event_names)), event_lead_days, color=colors, height=0.5)
for i, (detected, lead) in enumerate(zip(event_detected, event_lead_days)):
    status = f"Detected ({lead}d early)" if detected else "MISSED"
    ax.text(
        max(event_lead_days) * 0.02 + lead,
        i,
        status,
        va="center",
        fontsize=11,
        fontweight="bold",
        color="#1B5E20" if detected else "#B71C1C",
    )
ax.set_yticks(range(len(event_names)))
ax.set_yticklabels(event_names, fontsize=11)
ax.set_xlabel("Days of Early Warning")
ax.set_title("Event Detection: LSTM-AE Catches Regime Changes", fontsize=13)
ax.grid(True, alpha=0.3, axis="x")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_timeseries_event_detection.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Portfolio drawdown analysis ---
PORTFOLIO_VALUE = 100_000_000
portfolio_returns = daily_returns.mean(axis=1)
passive_cum = np.cumprod(1 + portfolio_returns) * PORTFOLIO_VALUE

anomaly_by_day = np.zeros(N_DAYS, dtype=bool)
for i, day in enumerate(seq_days):
    if is_anomaly[i] and day < N_DAYS:
        anomaly_by_day[day : min(day + 5, N_DAYS)] = True
adjusted_returns = np.where(anomaly_by_day, portfolio_returns * 0.2, portfolio_returns)
adjusted_cum = np.cumprod(1 + adjusted_returns) * PORTFOLIO_VALUE

passive_peak = np.maximum.accumulate(passive_cum)
passive_dd = (passive_cum - passive_peak) / passive_peak
adjusted_peak = np.maximum.accumulate(adjusted_cum)
adjusted_dd = (adjusted_cum - adjusted_peak) / adjusted_peak

fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(
    days,
    passive_cum / 1e6,
    color="#F44336",
    linewidth=1.5,
    label=f"Passive (S${passive_cum[-1]/1e6:.1f}M)",
)
axes[0].plot(
    days,
    adjusted_cum / 1e6,
    color="#4CAF50",
    linewidth=1.5,
    label=f"Adjusted (S${adjusted_cum[-1]/1e6:.1f}M)",
)
axes[0].set_ylabel("Portfolio Value (S$M)")
axes[0].set_title("S$100M SGX Portfolio: Passive vs Anomaly-Adjusted", fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[1].fill_between(
    days, passive_dd * 100, alpha=0.5, color="#F44336", label="Passive"
)
axes[1].fill_between(
    days, adjusted_dd * 100, alpha=0.5, color="#4CAF50", label="Adjusted"
)
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_xlabel("Trading Day")
axes[1].set_title("Maximum Drawdown Comparison", fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_timeseries_portfolio_drawdown.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Business Impact ---
passive_worst_loss = PORTFOLIO_VALUE * abs(passive_dd.min())
adjusted_worst_loss = PORTFOLIO_VALUE * abs(adjusted_dd.min())
dollar_saved = passive_worst_loss - adjusted_worst_loss

print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — SGX Regime Detection (S$100M Fund)")
print("=" * 64)
print(f"\nEvents detected: {sum(event_detected)}/{len(crisis_periods)}")
for i, (_, _, name) in enumerate(crisis_periods):
    status = f"Detected {event_lead_days[i]}d early" if event_detected[i] else "MISSED"
    print(f"  {name}: {status}")
print(f"\nPassive max drawdown:   {passive_dd.min()*100:.1f}%")
print(f"Adjusted max drawdown:  {adjusted_dd.min()*100:.1f}%")
print(f"Capital preserved:      S${dollar_saved:,.0f}")
print(f"\nFinal portfolio value:")
print(f"  Passive:              S${passive_cum[-1]:,.0f}")
print(f"  Anomaly-adjusted:     S${adjusted_cum[-1]:,.0f}")
print("=" * 64)



## REFLECTION


[x] Built an LSTM encoder-decoder for time-series data
  [x] Understood temporal order preservation via recurrent architecture
  [x] Visualised original vs reconstructed vibration patterns
  [x] Applied to SGX regime change detection with early warning
  [x] Built portfolio anomaly-adjusted strategy
  [x] Quantified S$ capital preserved at worst drawdown

  KEY INSIGHT: The LSTM encoder reads a sequence step-by-step,
  building a compressed summary that captures temporal patterns.
  When the market enters a regime the model has never seen (crisis),
  reconstruction error spikes — the model is saying "this sequence
  does not match any pattern I learned from normal markets." That
  signal, days before the worst of the crash, is worth millions
  in avoided drawdown.

  Next: 09_vae.py introduces probabilistic latent spaces...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

